# League of Legends Patch Notes + Champion Updates Scraper

这个 Notebook 用于：
1. 自动翻页抓取英雄联盟官网 Patch Notes 历史条目
2. 进入每一篇 Patch Note，解析其中提到的英雄和对应更新内容
3. 输出整合后的 DataFrame 和 CSV

目标页面：
`https://www.leagueoflegends.com/en-us/news/tags/patch-notes/`

In [1]:
import json
import re
import time
from typing import Dict, List
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.leagueoflegends.com"
PATCH_NOTES_URL = "https://www.leagueoflegends.com/en-us/news/tags/patch-notes/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

In [2]:
def get_next_data(url: str) -> Dict:
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    script = soup.find("script", id="__NEXT_DATA__")
    if not script or not script.string:
        raise ValueError(f"No __NEXT_DATA__ found for: {url}")

    return json.loads(script.string)


def extract_patch_items(next_data: Dict) -> List[Dict]:
    blades = next_data.get("props", {}).get("pageProps", {}).get("page", {}).get("blades", [])
    article_grid = next((b for b in blades if b.get("type") == "articleCardGrid"), None)
    if not article_grid:
        return []

    records = []
    for item in article_grid.get("items", []):
        title = (item.get("title") or "").strip()
        if not title or "patch" not in title.lower():
            continue

        rel_url = item.get("action", {}).get("payload", {}).get("url", "")
        if not rel_url:
            continue

        full_url = urljoin(BASE_URL, rel_url)
        published_at = item.get("publishedAt")

        raw_description = item.get("description")
        if isinstance(raw_description, str):
            summary = raw_description.strip()
        elif isinstance(raw_description, dict):
            # description 有时是富文本对象，转纯文本
            summary = re.sub(r"\s+", " ", BeautifulSoup(str(raw_description), "html.parser").get_text(" ", strip=True)).strip()
        else:
            summary = ""

        records.append(
            {
                "title": title,
                "url": full_url,
                "published_at": published_at,
                "summary": summary,
            }
        )

    return records


def crawl_patch_notes(max_pages: int = 50, sleep_sec: float = 0.25) -> pd.DataFrame:
    all_records = []
    seen_urls = set()

    for page in range(1, max_pages + 1):
        page_url = PATCH_NOTES_URL if page == 1 else f"{PATCH_NOTES_URL}?page={page}"

        try:
            next_data = get_next_data(page_url)
        except Exception as e:
            print(f"Stop at page {page}: {e}")
            break

        page_records = extract_patch_items(next_data)
        if not page_records:
            print(f"No records on page {page}, stop.")
            break

        new_count = 0
        for row in page_records:
            if row["url"] in seen_urls:
                continue
            seen_urls.add(row["url"])
            all_records.append(row)
            new_count += 1

        print(f"Page {page}: fetched={len(page_records)}, new={new_count}, total={len(all_records)}")

        # 如果新页面没有带来新数据，说明没有真实翻页数据了
        if new_count == 0:
            break

        time.sleep(sleep_sec)

    df_patches = pd.DataFrame(all_records)
    if not df_patches.empty:
        df_patches["published_at"] = pd.to_datetime(df_patches["published_at"], errors="coerce", utc=True)
        df_patches = df_patches.sort_values("published_at", ascending=False).reset_index(drop=True)

    return df_patches


patch_df = crawl_patch_notes(max_pages=100)
patch_df.head(20), len(patch_df)

Page 1: fetched=163, new=163, total=163
Page 2: fetched=163, new=0, total=163


(                                 title  \
 0   League of Legends Patch 26.8 Notes   
 1   League of Legends Patch 26.7 Notes   
 2   League of Legends Patch 26.6 Notes   
 3   League of Legends Patch 26.5 Notes   
 4   League of Legends Patch 26.4 Notes   
 5                     Patch 26.3 Notes   
 6                     Patch 26.2 Notes   
 7                     Patch 26.1 Notes   
 8                    Patch 25.24 Notes   
 9                    Patch 25.23 Notes   
 10                   Patch 25.22 Notes   
 11                   Patch 25.21 Notes   
 12                   Patch 25.20 Notes   
 13                   Patch 25.19 Notes   
 14                   Patch 25.18 Notes   
 15                   Patch 25.17 Notes   
 16                   Patch 25.16 Notes   
 17                   Patch 25.15 Notes   
 18                   Patch 25.14 Notes   
 19                   Patch 25.13 Notes   
 
                                                   url  \
 0   https://www.leagueoflegends.com/

In [3]:
CHAMPION_EXCLUDE_KEYWORDS = {
    "patch highlights",
    "related articles",
    "systems",
    "items",
    "runes",
    "aram",
    "arena",
    "ranked",
    "skins",
    "chromas",
    "mythic shop",
    "quality of life",
    "bugfixes",
    "bug fixes",
    "tft",
}


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()


def looks_like_champion_name(text: str) -> bool:
    t = clean_text(text)
    if not t:
        return False

    lower = t.lower()
    if lower in CHAMPION_EXCLUDE_KEYWORDS:
        return False

    if len(t) > 40:
        return False

    if any(sym in t for sym in ["/", "(", ")", "[", "]", "{", "}", ":"]):
        return False

    # 能力小节常见格式："Q - Name"
    if " - " in t:
        return False

    allowed = re.compile(r"^[A-Za-z0-9'&.\- ]+$")
    if not allowed.match(t):
        return False

    return True


def parse_champion_updates_from_article(url: str) -> List[Dict]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    main = soup.find("main") or soup

    updates = []
    champion_lines: Dict[str, List[str]] = {}

    in_champion_section = False
    current_champion = None

    stream_tags = main.find_all(["h2", "h3", "h4", "p", "li"])

    for tag in stream_tags:
        tag_name = tag.name.lower()
        text = clean_text(tag.get_text(" ", strip=True))
        if not text:
            continue

        if tag_name == "h2":
            lower = text.lower()

            # 进入冠军改动区域（如：Champions / Champion Buffs / Champion Nerfs）
            if "champion" in lower:
                in_champion_section = True
                current_champion = None
                continue

            # 离开冠军改动区域
            if in_champion_section:
                in_champion_section = False
                current_champion = None

            continue

        if not in_champion_section:
            continue

        # 英雄名以 h3 为主，能明显减少把 "Base Stats" 这类小节标题误判成英雄
        if tag_name == "h3" and looks_like_champion_name(text):
            current_champion = text
            champion_lines.setdefault(current_champion, [])
            continue

        if current_champion and tag_name in {"h4", "p", "li"}:
            # 排除重复英雄名行
            if text == current_champion:
                continue
            champion_lines[current_champion].append(text)

    for champion, lines in champion_lines.items():
        # 去重并保序
        deduped = list(dict.fromkeys([clean_text(x) for x in lines if clean_text(x)]))
        if not deduped:
            continue

        updates.append(
            {
                "champion": champion,
                "champion_update": "\n".join(deduped),
            }
        )

    return updates


MAX_ARTICLES = None  # 调试时可改成 5 或 10，加快运行
source_df = patch_df if MAX_ARTICLES is None else patch_df.head(MAX_ARTICLES)

rows = []
for idx, row in source_df.iterrows():
    print(f"[{idx + 1}/{len(source_df)}] Parsing: {row['title']}")
    champion_updates = parse_champion_updates_from_article(row["url"])

    if not champion_updates:
        continue

    for cu in champion_updates:
        rows.append(
            {
                "patch_title": row["title"],
                "patch_url": row["url"],
                "published_at": row["published_at"],
                "patch_summary": row.get("summary", ""),
                "champion": cu["champion"],
                "champion_update": cu["champion_update"],
            }
        )

champion_updates_df = pd.DataFrame(rows)
if not champion_updates_df.empty:
    champion_updates_df = champion_updates_df.sort_values(
        by=["published_at", "patch_title", "champion"], ascending=[False, True, True]
    ).reset_index(drop=True)

champion_updates_df.head(30), len(champion_updates_df)

[1/163] Parsing: League of Legends Patch 26.8 Notes
[2/163] Parsing: League of Legends Patch 26.7 Notes
[3/163] Parsing: League of Legends Patch 26.6 Notes
[4/163] Parsing: League of Legends Patch 26.5 Notes
[5/163] Parsing: League of Legends Patch 26.4 Notes
[6/163] Parsing: Patch 26.3 Notes
[7/163] Parsing: Patch 26.2 Notes
[8/163] Parsing: Patch 26.1 Notes
[9/163] Parsing: Patch 25.24 Notes
[10/163] Parsing: Patch 25.23 Notes
[11/163] Parsing: Patch 25.22 Notes
[12/163] Parsing: Patch 25.21 Notes
[13/163] Parsing: Patch 25.20 Notes
[14/163] Parsing: Patch 25.19 Notes
[15/163] Parsing: Patch 25.18 Notes
[16/163] Parsing: Patch 25.17 Notes
[17/163] Parsing: Patch 25.16 Notes
[18/163] Parsing: Patch 25.15 Notes
[19/163] Parsing: Patch 25.14 Notes
[20/163] Parsing: Patch 25.13 Notes
[21/163] Parsing: Patch 25.12 Notes
[22/163] Parsing: Patch 25.11 Notes
[23/163] Parsing: Patch 25.10 Notes
[24/163] Parsing: Patch 25.09 Notes
[25/163] Parsing: Patch 25.08 Notes
[26/163] Parsing: Patch 25.

(                           patch_title  \
 0   League of Legends Patch 26.8 Notes   
 1   League of Legends Patch 26.8 Notes   
 2   League of Legends Patch 26.8 Notes   
 3   League of Legends Patch 26.8 Notes   
 4   League of Legends Patch 26.8 Notes   
 5   League of Legends Patch 26.8 Notes   
 6   League of Legends Patch 26.8 Notes   
 7   League of Legends Patch 26.8 Notes   
 8   League of Legends Patch 26.8 Notes   
 9   League of Legends Patch 26.8 Notes   
 10  League of Legends Patch 26.7 Notes   
 11  League of Legends Patch 26.7 Notes   
 12  League of Legends Patch 26.7 Notes   
 13  League of Legends Patch 26.7 Notes   
 14  League of Legends Patch 26.7 Notes   
 15  League of Legends Patch 26.7 Notes   
 16  League of Legends Patch 26.7 Notes   
 17  League of Legends Patch 26.7 Notes   
 18  League of Legends Patch 26.7 Notes   
 19  League of Legends Patch 26.6 Notes   
 20  League of Legends Patch 26.6 Notes   
 21  League of Legends Patch 26.6 Notes   
 22  League

In [4]:
patch_output_path = "lol_patch_notes_index.csv"
champion_output_path = "lol_patch_notes_champion_updates.csv"

patch_df.to_csv(patch_output_path, index=False, encoding="utf-8")
champion_updates_df.to_csv(champion_output_path, index=False, encoding="utf-8")

print(f"Saved patch index: {len(patch_df)} rows -> {patch_output_path}")
print(f"Saved champion updates: {len(champion_updates_df)} rows -> {champion_output_path}")

champion_updates_df.head(20)

Saved patch index: 163 rows -> lol_patch_notes_index.csv
Saved champion updates: 2540 rows -> lol_patch_notes_champion_updates.csv


,patch_title,patch_url,published_at,patch_summary,champion,champion_update
0,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Dr. Mundo,Dr. Mundo has been able to more consistently r...
1,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Hwei,Hwei has remained on the weaker side since the...
2,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Karma,Last patch we made some changes to Karma to re...
3,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lillia,Our favorite bouncing deer has been a bit less...
4,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lucian,Lucian has a unique role as an AD Carry that d...
5,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Mel,"Since our recent adjustments to Mel, she’s bee..."
6,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Tahm Kench,"As a champ in both Top and Support, Tahm Kench..."
7,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Yuumi,We’re implementing a couple bugfixes to Yuumi’...
8,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Zilean,As a quality of life change we’re looking to m...
9,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Zyra,Since the dawn of time Zyra’s have been accide...


## Data Cleaning

本节完成两件事：
1. 将发布时间统一到 `day` 粒度
2. 将 `champion_update` 的明细按项目拆分为多列（缺失值保留为 `NaN`）

In [5]:
def is_update_header(line: str) -> bool:
    line = clean_text(line)
    if not line:
        return False

    # 常见技能/属性标题
    if re.match(r"^(Base Stats|Passive|Q\s*-\s*|W\s*-\s*|E\s*-\s*|R\s*-\s*)", line, flags=re.I):
        return True

    # 兜底：短标题且不像完整句子
    if len(line) <= 45 and len(line.split()) <= 8 and not re.search(r"[.!?]$", line):
        if ":" not in line and ";" not in line:
            return True

    return False


def normalize_col_name(name: str) -> str:
    s = name.lower().strip()
    s = re.sub(r"\s*-\s*", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return f"detail_{s}" if s else "detail_unknown"


def split_update_to_sections(update_text: str) -> Dict[str, str]:
    if not isinstance(update_text, str) or not update_text.strip():
        return {}

    lines = [clean_text(x) for x in update_text.split("\n") if clean_text(x)]
    if not lines:
        return {}

    sections: Dict[str, List[str]] = {}
    current_header = None

    for line in lines:
        if is_update_header(line):
            current_header = line
            sections.setdefault(current_header, [])
            continue

        if current_header is None:
            current_header = "General"
            sections.setdefault(current_header, [])

        sections[current_header].append(line)

    # 将每个 section 的多行明细合并成单字符串
    output = {}
    for k, v in sections.items():
        merged = " ".join(v).strip()
        output[k] = merged if merged else None

    return output


clean_df = champion_updates_df.copy()

# 1) 时间统一到 day 粒度
clean_df["date"] = pd.to_datetime(clean_df["published_at"], errors="coerce", utc=True).dt.floor("D")
clean_df["date"] = clean_df["date"].dt.tz_localize(None)

# 2) update 明细拆列
section_dicts = clean_df["champion_update"].apply(split_update_to_sections)
section_wide = pd.DataFrame(section_dicts.tolist())

# 原 section 名 -> 标准列名
rename_map = {c: normalize_col_name(c) for c in section_wide.columns}
section_wide = section_wide.rename(columns=rename_map)

# 同名列（标准化后冲突）做聚合合并
if section_wide.columns.duplicated().any():
    section_wide = section_wide.T.groupby(level=0).agg(
        lambda col: next((x for x in col if isinstance(x, str) and x.strip()), None)
    ).T

clean_df = pd.concat(
    [clean_df.drop(columns=["champion_update"]), section_wide],
    axis=1,
)

# 常用字段放前面
base_cols = ["date", "published_at", "patch_title", "champion", "patch_url", "patch_summary"]
detail_cols = [c for c in clean_df.columns if c.startswith("detail_")]
other_cols = [c for c in clean_df.columns if c not in base_cols + detail_cols]
clean_df = clean_df[base_cols + detail_cols + other_cols]

clean_output_path = "lol_patch_notes_champion_updates_cleaned.csv"
clean_df.to_csv(clean_output_path, index=False, encoding="utf-8")

print(f"Saved cleaned rows: {len(clean_df)} -> {clean_output_path}")
print(f"Detail columns: {len(detail_cols)}")
clean_df.head(20)

Saved cleaned rows: 2540 -> lol_patch_notes_champion_updates_cleaned.csv
Detail columns: 1582


,date,published_at,patch_title,champion,patch_url,patch_summary,detail_0_25_seconds,detail_10,detail_110,detail_1_25_seconds,...,detail_yone_abilities_rundown,detail_yone_biography,detail_yone_champion_insights,detail_yorick,detail_zeal,detail_zeri_abilities_rundown,detail_zeri_champion_bio,detail_zeri_champion_insights,detail_zeri_champion_trailer,detail_zilean
0,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Dr. Mundo,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Hwei,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Karma,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Lillia,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Lucian,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Mel,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Tahm Kench,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Yuumi,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
8,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Zilean,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Zyra,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


## 分类清洗（按改动类型分栏）

你要求的是“技能类改动放同一栏，并在栏内保留具体改动文本”。

此单元会输出这些列：
- `base_stats_changes`
- `passive_changes`
- `q_changes` / `w_changes` / `e_changes` / `r_changes`
- `other_changes`（其他非技能项）

若该记录没有对应类别改动，则该列为 `NaN`。

In [6]:
import numpy as np


def _is_header_line(line: str) -> bool:
    line = clean_text(line)
    if not line:
        return False

    # 常见标题形式：Base Stats / Passive / Q - xxx / W - xxx / E - xxx / R - xxx
    if re.match(r"^(Base Stats|Passive|Q\s*-\s*.+|W\s*-\s*.+|E\s*-\s*.+|R\s*-\s*.+)$", line, flags=re.I):
        return True

    return False


def _bucket_from_header(header: str) -> str:
    h = clean_text(header).lower()

    if h.startswith("base stats"):
        return "base_stats_changes"
    if h.startswith("passive"):
        return "passive_changes"
    if h.startswith("q -"):
        return "q_changes"
    if h.startswith("w -"):
        return "w_changes"
    if h.startswith("e -"):
        return "e_changes"
    if h.startswith("r -"):
        return "r_changes"

    return "other_changes"


def categorize_update_text(update_text: str) -> Dict[str, object]:
    buckets = {
        "base_stats_changes": [],
        "passive_changes": [],
        "q_changes": [],
        "w_changes": [],
        "e_changes": [],
        "r_changes": [],
        "other_changes": [],
    }

    if not isinstance(update_text, str) or not update_text.strip():
        return {k: np.nan for k in buckets.keys()}

    lines = [clean_text(x) for x in update_text.split("\n") if clean_text(x)]
    current_header = None
    current_bucket = "other_changes"

    for line in lines:
        if _is_header_line(line):
            current_header = line
            current_bucket = _bucket_from_header(line)
            # 把标题本身也保留在栏位里，便于回看
            buckets[current_bucket].append(f"[{current_header}]")
            continue

        buckets[current_bucket].append(line)

    # 每个类别合并成单列字符串；空值设为 NaN
    result = {}
    for k, v in buckets.items():
        merged = "\n".join(v).strip()
        result[k] = merged if merged else np.nan

    return result


categorized_df = champion_updates_df.copy()

# 1) 日期按 day 粒度
categorized_df["date"] = pd.to_datetime(categorized_df["published_at"], errors="coerce", utc=True).dt.floor("D")
categorized_df["date"] = categorized_df["date"].dt.tz_localize(None)

# 2) champion_update 分类拆列
cat_cols = categorized_df["champion_update"].apply(categorize_update_text).apply(pd.Series)
categorized_df = pd.concat([categorized_df, cat_cols], axis=1)

# 字段排序
ordered_cols = [
    "date",
    "published_at",
    "patch_title",
    "champion",
    "patch_url",
    "patch_summary",
    "base_stats_changes",
    "passive_changes",
    "q_changes",
    "w_changes",
    "e_changes",
    "r_changes",
    "other_changes",
    "champion_update",
]
ordered_cols = [c for c in ordered_cols if c in categorized_df.columns]
categorized_df = categorized_df[ordered_cols]

categorized_output_path = "lol_patch_notes_champion_updates_categorized.csv"
categorized_df.to_csv(categorized_output_path, index=False, encoding="utf-8")

print(f"Saved categorized rows: {len(categorized_df)} -> {categorized_output_path}")
categorized_df.head(20)

Saved categorized rows: 2540 -> lol_patch_notes_champion_updates_categorized.csv


,date,published_at,patch_title,champion,patch_url,patch_summary,base_stats_changes,passive_changes,q_changes,w_changes,e_changes,r_changes,other_changes,champion_update
0,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Dr. Mundo,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,[Q - Infected Bonesaw]\nMonster Damage Cap : 3...,NaN,[E - Blunt Force Trauma]\nMonster Damage Cap: ...,NaN,Dr. Mundo has been able to more consistently r...,Dr. Mundo has been able to more consistently r...
1,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Hwei,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,NaN,NaN,NaN,Hwei has remained on the weaker side since the...,Hwei has remained on the weaker side since the...
2,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Karma,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",[Base Stats]\nBase AD : 51 ⇒ 49\nAD Growth : 3...,NaN,NaN,NaN,[E - Inspire]\nMana Cost : 50 / 55 / 60 / 65 /...,NaN,Last patch we made some changes to Karma to re...,Last patch we made some changes to Karma to re...
3,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Lillia,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,NaN,NaN,NaN,Our favorite bouncing deer has been a bit less...,Our favorite bouncing deer has been a bit less...
4,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Lucian,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,NaN,[E - Relentless Pursuit]\nCooldown : 18 / 17 /...,NaN,Lucian has a unique role as an AD Carry that d...,Lucian has a unique role as an AD Carry that d...
5,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Mel,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,[Q - Radiant Volley]\nInitial Explosion Damage...,[W - Rebuttal]\nCooldown : 35 / 32 / 29 / 26 /...,NaN,NaN,"Since our recent adjustments to Mel, she’s bee...","Since our recent adjustments to Mel, she’s bee..."
6,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Tahm Kench,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,NaN,NaN,NaN,"As a champ in both Top and Support, Tahm Kench...","As a champ in both Top and Support, Tahm Kench..."
7,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Yuumi,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,NaN,NaN,[R - Final Chapter]\nHeal Per Wave : 30 / 40 /...,We’re implementing a couple bugfixes to Yuumi’...,We’re implementing a couple bugfixes to Yuumi’...
8,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Zilean,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,[W - Rewind]\nW Level Up : No longer can level...,NaN,NaN,As a quality of life change we’re looking to m...,As a quality of life change we’re looking to m...
9,2026-04-14,2026-04-14 18:00:00+00:00,League of Legends Patch 26.8 Notes,Zyra,https://www.leagueoflegends.com/en-us/news/gam...,"{'type': 'html', 'body': 'Let’s get groovy wit...",NaN,NaN,NaN,[W - Rampant Growth]\nW Level Up : No longer c...,NaN,NaN,Since the dawn of time Zyra’s have been accide...,Since the dawn of time Zyra’s have been accide...


In [7]:
champion_updates_df.head(100)

,patch_title,patch_url,published_at,patch_summary,champion,champion_update
0,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Dr. Mundo,Dr. Mundo has been able to more consistently r...
1,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Hwei,Hwei has remained on the weaker side since the...
2,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Karma,Last patch we made some changes to Karma to re...
3,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lillia,Our favorite bouncing deer has been a bit less...
4,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lucian,Lucian has a unique role as an AD Carry that d...
...,...,...,...,...,...,...
95,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Tryndamere,Base Stats\nAttack Damage Growth : 4 ⇒ 4.5
96,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Varus,Base Stats\nArmor Growth : 4.6 ⇒ 4\nW - Blight...
97,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Vi,Base Stats\nHealth Growth : 99 ⇒ 105\nAttack D...
98,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Volibear,Q - Thundering Smash\nBonus Physical Damage : ...


In [8]:
# 统一导出（推荐运行这个单元，避免文件名混淆）
# 说明：
# - patch_df: patch note 索引（每篇一行）
# - champion_updates_df: 原始英雄改动明细（每个英雄一行）
# - categorized_df: 分类清洗后的明细

export_files = {
    # 为兼容旧文件名，这里把 lol_patch_notes.csv 也导出成“全量索引”
    "lol_patch_notes.csv": patch_df,
    "lol_patch_notes_index.csv": patch_df,
    "lol_patch_notes_raw.csv": champion_updates_df,
    "lol_patch_notes_champion_updates.csv": champion_updates_df,
}

# 如果你已经运行了分类单元，也一并导出
if "categorized_df" in globals():
    export_files["lol_patch_notes_champion_updates_categorized.csv"] = categorized_df

for file_name, data in export_files.items():
    data.to_csv(file_name, index=False, encoding="utf-8")
    print(f"Saved {file_name}: rows={len(data)}")

print("Done. You can open lol_patch_notes.csv safely now.")

Saved lol_patch_notes.csv: rows=163
Saved lol_patch_notes_index.csv: rows=163
Saved lol_patch_notes_raw.csv: rows=2540
Saved lol_patch_notes_champion_updates.csv: rows=2540
Saved lol_patch_notes_champion_updates_categorized.csv: rows=2540
Done. You can open lol_patch_notes.csv safely now.


In [9]:
# 导出原始抓取数据（未做分类清洗）
raw_output_path = "lol_patch_notes_raw.csv"

# champion_updates_df 为逐篇解析后的原始明细（包含 champion_update 原始文本）
champion_updates_df.to_csv(raw_output_path, index=False, encoding="utf-8")

print(f"Saved raw rows: {len(champion_updates_df)} -> {raw_output_path}")
champion_updates_df.head(100)

Saved raw rows: 2540 -> lol_patch_notes_raw.csv


,patch_title,patch_url,published_at,patch_summary,champion,champion_update
0,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Dr. Mundo,Dr. Mundo has been able to more consistently r...
1,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Hwei,Hwei has remained on the weaker side since the...
2,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Karma,Last patch we made some changes to Karma to re...
3,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lillia,Our favorite bouncing deer has been a bit less...
4,League of Legends Patch 26.8 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-04-14 18:00:00+00:00,"{'type': 'html', 'body': 'Let’s get groovy wit...",Lucian,Lucian has a unique role as an AD Carry that d...
...,...,...,...,...,...,...
95,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Tryndamere,Base Stats\nAttack Damage Growth : 4 ⇒ 4.5
96,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Varus,Base Stats\nArmor Growth : 4.6 ⇒ 4\nW - Blight...
97,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Vi,Base Stats\nHealth Growth : 99 ⇒ 105\nAttack D...
98,Patch 26.3 Notes,https://www.leagueoflegends.com/en-us/news/gam...,2026-02-03 19:00:00+00:00,"{'type': 'html', 'body': 'Celebrate the new ye...",Volibear,Q - Thundering Smash\nBonus Physical Damage : ...
